In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from AttackInfo import attack_info, Sensors, Actuators

In [37]:
df_normal = pd.read_parquet('../../Dataset/SWaT_Dataset_Normal_v1.parquet')
df_attack  = pd.read_parquet('../../Dataset/SWaT_Dataset_Attack_v1.parquet')

In [39]:
# 格式化顯示攻擊清單
rows = []
for atk_id, info in attack_info.items():
    rows.append({
        'Attack#': atk_id,
        'Category': info['cat'],
        'Type': info['type'],
        'Actual Change': '✅ Yes' if info['actual_change'] else '❌ No',
        'Physical Impact': '✅ Yes' if info['impact'] else '❌ No',
        'Attack Point(s)': ', '.join(info['point']) if info['point'] else '—',
        'Stage(s)': str(info['stage']) if info['stage'] else '—',
        'Description': info['desc'],
    })

atk_df = pd.DataFrame(rows)

# 統計摘要
print("=== 攻擊類型分佈 ===")
print(atk_df['Category'].value_counts().to_string())
print()
print("=== 攻擊手法分佈 ===")
print(atk_df['Type'].value_counts().to_string())
print()
print("=== Actual Change 分佈 ===")
print(atk_df['Actual Change'].value_counts().to_string())

display(atk_df.set_index('Attack#'))

=== 攻擊類型分佈 ===
Category
SSSP    22
SSMP     7
MSSP     4
MSMP     3

=== 攻擊手法分佈 ===
Type
sensor      17
actuator    11
mixed        8

=== Actual Change 分佈 ===
Actual Change
✅ Yes    19
❌ No     17


,Category,Type,Actual Change,Physical Impact,Attack Point(s),Stage(s),Description
Attack#,,,,,,,
1,SSSP,actuator,✅ Yes,✅ Yes,MV101,[1],Open MV-101
2,SSSP,actuator,✅ Yes,✅ Yes,P102,[1],Turn on P-102
3,SSSP,sensor,❌ No,✅ Yes,LIT101,[1],Spoof LIT-101 +1mm/s
4,SSSP,actuator,✅ Yes,❌ No,MV504,[5],Open MV-504
6,SSSP,sensor,❌ No,✅ Yes,AIT202,[2],Set AIT-202 = 6
7,SSSP,sensor,❌ No,✅ Yes,LIT301,[3],Spoof LIT-301 > HH
8,SSSP,sensor,❌ No,✅ Yes,DPIT301,[3],Set DPIT-301 > 40kPa
10,SSSP,sensor,❌ No,✅ Yes,FIT401,[4],Set FIT-401 < 0.7
11,SSSP,sensor,❌ No,✅ Yes,FIT401,[4],Set FIT-401 = 0


In [43]:
# 9.1 建立清潔標籤（排除 NPI 攻擊）
NPI_labels = {'Attack5', 'Attack9', 'Attack12', 'Attack15', 'Attack18'}

df_attack = df_attack.copy()
df_attack['Label_Clean'] = df_attack['Detailed_Label'].apply(
    lambda x: 0 if (x == 'Normal' or x in NPI_labels) else 1
)

# 攻擊型態標籤
actuator_attack_ids = {k for k, v in attack_info.items() if v['actual_change'] and v['type'] != 'none'}
sensor_attack_ids   = {k for k, v in attack_info.items() if not v['actual_change'] and v['type'] != 'none'}

def get_attack_id(label_str):
    if not label_str.startswith('Attack'):
        return None
    try:
        return int(label_str.replace('Attack', ''))
    except:
        return None

df_attack['Label_Actuator'] = df_attack['Detailed_Label'].apply(
    lambda x: 1 if get_attack_id(x) in actuator_attack_ids else 0
)
df_attack['Label_Sensor'] = df_attack['Detailed_Label'].apply(
    lambda x: 1 if get_attack_id(x) in sensor_attack_ids else 0
)

print("=== 標籤統計比較 ===")
label_comparison = pd.DataFrame({
    'Label (原始)':          df_attack['Label'].value_counts(),
    'Label_Clean (排除NPI)': df_attack['Label_Clean'].value_counts(),
    'Label_Actuator':        df_attack['Label_Actuator'].value_counts(),
    'Label_Sensor':          df_attack['Label_Sensor'].value_counts(),
}).fillna(0).astype(int)
print(label_comparison)
print()

npi_count = df_attack['Detailed_Label'].isin(NPI_labels).sum()
print(f"因排除 NPI 攻擊而移除的攻擊標籤筆數：{npi_count}")
print(f"Label=1 → Label_Clean=1 的比例：{df_attack['Label_Clean'].sum() / df_attack['Label'].sum() * 100:.1f}%")

=== 標籤統計比較 ===
   Label (原始)  Label_Clean (排除NPI)  Label_Actuator  Label_Sensor
0      396034               396034          404263        441690
1       53885                53885           45656          8229

因排除 NPI 攻擊而移除的攻擊標籤筆數：0
Label=1 → Label_Clean=1 的比例：100.0%


In [44]:
# 9.2 攻擊點偵測標籤（僅感測器攻擊點欄位有顯著異常才標記）
# 方法：對每筆 Label=1 的資料，檢查該攻擊對應的感測器攻擊點是否超出 Normal 3σ

normal_mean = df_normal[Sensors].mean()
normal_std  = df_normal[Sensors].std()

def is_sensor_anomalous(row, atk_id):
    if atk_id not in attack_info:
        return 0
    sensor_points = [p for p in attack_info[atk_id]['point'] if p in Sensors]
    if not sensor_points:
        return 0
    for sp in sensor_points:
        z = abs(row[sp] - normal_mean[sp]) / (normal_std[sp] + 1e-9)
        if z > 3:
            return 1
    return 0

# 只對攻擊行計算（執行時間較長）
attack_rows_idx = df_attack[df_attack['Label'] == 1].index
atk_id_series = df_attack.loc[attack_rows_idx, 'Detailed_Label'].apply(
    lambda x: get_attack_id(x)
)

# 向量化計算（避免逐行）
label_anomaly = np.zeros(len(df_attack), dtype=int)
for atk_id in attack_info:
    label_str = f'Attack{atk_id}'
    mask = df_attack['Detailed_Label'] == label_str
    sensor_points = [p for p in attack_info[atk_id]['point'] if p in Sensors]
    if not sensor_points or not mask.any():
        continue
    z = ((df_attack.loc[mask, sensor_points] - normal_mean[sensor_points])
         / (normal_std[sensor_points] + 1e-9)).abs()
    anomalous = (z > 3).any(axis=1)
    label_anomaly[mask.values] = anomalous.values.astype(int)

df_attack['Label_Anomaly'] = label_anomaly

print("=== Label_Anomaly 統計 ===")
print(df_attack[df_attack['Label']==1]['Label_Anomaly'].value_counts())
print(f"\n在 Label=1 的資料中，感測器攻擊點確實超出 3σ 的比例：{df_attack[df_attack['Label']==1]['Label_Anomaly'].mean()*100:.1f}%")
print()
print("(含 NPI 攻擊與純執行器攻擊，感測器攻擊點為空，故這些攻擊的 Label_Anomaly = 0)")

=== Label_Anomaly 統計 ===
Label_Anomaly
0    47081
1     6804
Name: count, dtype: int64

在 Label=1 的資料中，感測器攻擊點確實超出 3σ 的比例：12.6%

(含 NPI 攻擊與純執行器攻擊，感測器攻擊點為空，故這些攻擊的 Label_Anomaly = 0)


In [41]:
print("=== Normal_v1 ===")
print(f"Shape : {df_normal.shape}")
print(f"Time  : {df_normal['Timestamp'].min()}  →  {df_normal['Timestamp'].max()}")
print(f"Labels: {dict(df_normal['Label'].value_counts())}")
print()
print("=== Attack_v1 ===")
print(f"Shape : {df_attack.shape}")
print(f"Time  : {df_attack['Timestamp'].min()}  →  {df_attack['Timestamp'].max()}")
print(f"Labels: {dict(df_attack['Label'].value_counts())}")
print(f"Detailed_Label 種類數: {df_attack['Detailed_Label'].nunique()}")

=== Normal_v1 ===
Shape : (495000, 54)
Time  : 2015-12-22 16:30:00  →  2015-12-28 09:59:59
Labels: {0: np.int64(495000)}

=== Attack_v1 ===
Shape : (449919, 57)
Time  : 2015-12-28 10:00:00  →  2016-01-02 14:59:59
Labels: {0: np.int64(396034), 1: np.int64(53885)}
Detailed_Label 種類數: 37


In [45]:
# 11.1 合併兩個資料集
df_normal_proc = df_normal.copy()
df_normal_proc['Label_Clean']    = 0
df_normal_proc['Label_Actuator'] = 0
df_normal_proc['Label_Sensor']   = 0
df_normal_proc['Label_Anomaly']  = 0
df_normal_proc['source'] = 'normal'

shared_cols = [c for c in df_normal_proc.columns if c != 'source']
df_attack_proc = df_attack[shared_cols].copy()
df_attack_proc['source'] = 'attack'

df_combined = pd.concat([df_normal_proc, df_attack_proc], ignore_index=True)
df_combined = df_combined.sort_values('Timestamp').reset_index(drop=True)

print(f"合併後資料集 shape: {df_combined.shape}")
print(f"時間範圍: {df_combined['Timestamp'].min()}  →  {df_combined['Timestamp'].max()}")
print(f"\nLabel 分佈:")
print(df_combined['Label_Clean'].value_counts().to_string())

合併後資料集 shape: (944919, 59)
時間範圍: 2015-12-22 16:30:00  →  2016-01-02 14:59:59

Label 分佈:
Label_Clean
0    891034
1     53885


In [ ]:
# 11.4 最終資料集摘要
print("=" * 60)
print("         SWaT 資料集分析摘要")
print("=" * 60)
print()
print("【資料規模】")
print(f"  Normal_v1 : {len(df_normal):>8,} 筆  ({df_normal['Timestamp'].min().date()} ~ {df_normal['Timestamp'].max().date()})")
print(f"  Attack_v1 : {len(df_attack):>8,} 筆  ({df_attack['Timestamp'].min().date()} ~ {df_attack['Timestamp'].max().date()})")
print(f"  合計      : {len(df_normal)+len(df_attack):>8,} 筆")
print()
print("【特徵欄位】")
print(f"  感測器 (Sensors)   : {len(Sensors)} 個（連續型）")
print(f"  致動器 (Actuators) : {len(Actuators)} 個（類別型）")
print()
print("【攻擊事件】")
total_attacks = len(attack_info)
npi_attacks_n   = sum(1 for v in attack_info.values() if v['cat'] == 'NPI')
actuator_n = sum(1 for v in attack_info.values() if v['actual_change'])
sensor_n   = sum(1 for v in attack_info.values() if not v['actual_change'] and v['type'] != 'none')
print(f"  總攻擊數    : {total_attacks}")
print(f"  NPI 攻擊    : {npi_attacks_n} （標籤中有雜訊）")
print(f"  執行器攻擊  : {actuator_n} （Actual Change = Yes）")
print(f"  感測器欺騙  : {sensor_n} （Actual Change = No）")
print()
print("【Attack_v1 標籤統計】")
print(f"  原始 Label=1 筆數       : {df_attack['Label'].sum():>7,} ({df_attack['Label'].mean()*100:.2f}%)")
print(f"  清潔 Label_Clean=1 筆數 : {df_attack['Label_Clean'].sum():>7,} ({df_attack['Label_Clean'].mean()*100:.2f}%)")
print(f"  感測器攻擊點超 3σ 比例  : {df_attack[df_attack['Label']==1]['Label_Anomaly'].mean()*100:.1f}%")
print()
print("【前處理建議】")
print("  1. 感測器：StandardScaler 以 Normal_v1 擬合")
print("  2. 致動器：保持整數編碼 (0/1/2 或 1/2)")
print("  3. 標籤：建議使用 Label_Clean（排除 NPI 攻擊）")
print("  4. 滑動窗口：注意 Attack_v1 的 ~82 秒斷點")
print("  5. 訓練集：Normal_v1；測試集：Attack_v1（時間無重疊）")